# 03. 병원 내 사망 예측 — TF-IDF + Logistic Regression / Random Forest (Exp A·B)

논문 본 실험(Bio_ClinicalBERT 임베딩)과 별도로, 같은 MIMIC-IV-Note 퇴원 요약지에서 **TF-IDF(1–2gram, 5,000 피처) + LR / RF** 로 A/B 전처리를 비교한 실험.

- 타겟: `admissions.hospital_expire_flag` (병원 내 사망)
- 코호트: 퇴원 요약지가 있는 입원 331,732건 (train 265,385 / test 66,347)
- 서버 실행 결과: **Exp A 전체 텍스트** LR AUROC 0.9995 · RF 0.9994 / **Exp B 결과 섹션 제거** LR 0.9727 · RF 0.9429

출력 셀은 MIMIC 텍스트가 포함되므로 모두 비웠다.

## Exp A — 전체 텍스트

In [ ]:
import pandas as pd
import sys
import warnings
import traceback  # 👈 디버깅을 위해 추가

warnings.filterwarnings('ignore')

# ============================================================================
# 1단계: 데이터 로드 및 코호트 생성 (수정본)
# ============================================================================

# (함수 정의는 동일합니다)
def load_admissions(filepath):
    """(admissions.csv) 입원 정보 로드 (메모리 최적화)"""
    print(f"Loading admissions data from {filepath}...")
    usecols = ['subject_id', 'hadm_id', 'hospital_expire_flag', 
               'admission_type', 'admission_location', 'insurance']
    dtypes = {'subject_id': 'int32', 'hadm_id': 'int32', 'hospital_expire_flag': 'int8'}
    
    # ❗️ FileNotFoundError를 잡기 위해 try...except를 함수 내부에도 추가
    try:
        df = pd.read_csv(filepath, usecols=usecols, dtype=dtypes)
        print(f"  ✓ Loaded {len(df):,} admissions")
        return df
    except FileNotFoundError:
        print(f"\n[!!! 진짜 오류 !!!] {filepath} 파일을 찾을 수 없습니다.")
        print("경로를 확인하거나 PhysioNet에서 'core' 모듈을 다운로드하세요.")
        sys.exit()
    except Exception as e:
        print(f"\n[!!! 진짜 오류 !!!] {filepath} 로드 중 문제 발생:")
        raise e # 오류를 다시 발생시켜 상위 except에서 잡도록 함

def load_discharge(filepath):
    """(discharge.csv) 퇴원 요약본 로드 (메모리 최적화)"""
    print(f"Loading discharge notes from {filepath}...")
    usecols = ['subject_id', 'hadm_id', 'text']
    dtypes = {'subject_id': 'int32', 'hadm_id': 'int32', 'text': 'str'}
    
    # ❗️ FileNotFoundError를 잡기 위해 try...except를 함수 내부에도 추가
    try:
        df = pd.read_csv(filepath, usecols=usecols, dtype=dtypes)
        print(f"  ✓ Loaded {len(df):,} discharge summaries")
        return df
    except FileNotFoundError:
        print(f"\n[!!! 진짜 오류 !!!] {filepath} 파일을 찾을 수 없습니다.")
        print("경로를 확인하거나 PhysioNet에서 'note' 모듈을 다운로드하세요.")
        sys.exit()
    except Exception as e:
        print(f"\n[!!! 진짜 오류 !!!] {filepath} 로드 중 문제 발생:")
        raise e

def create_cohort(admissions_path, discharge_path):
    """코호트 생성: (admissions.csv + discharge.csv) 병합"""
    df_adm = load_admissions(admissions_path)
    df_notes = load_discharge(discharge_path)
    
    print("\nMerging admissions and discharge summaries...")
    cohort = df_notes.merge(df_adm, on=['subject_id', 'hadm_id'], how='inner')
    
    cohort = cohort.drop_duplicates(subset=['hadm_id'], keep='first')
    cohort = cohort.dropna(subset=['text', 'hospital_expire_flag'])
    cohort = cohort[cohort['text'].str.strip().str.len() > 100]
    
    print(f"\n  ✓ Final cohort created: {len(cohort):,} patients")
    print(f"  ✓ Mortality rate: {cohort['hospital_expire_flag'].mean():.2%}")
    return cohort

# ============================================================================
# 실행 (👈 여기가 핵심입니다)
# ============================================================================
try:
    print("\n[1단계] 데이터 로드 및 코호트 생성 시작...")
    
    # ⚠️ 이곳의 파일 경로를 실제 파일 위치로 수정해주세요.
    ADMISSIONS_FILE_PATH = "data/physionet.org/files/mimiciv/3.1/hosp/admissions.csv"
    DISCHARGE_FILE_PATH = "data/physionet.org/files/mimic-iv-note/2.2/note/discharge.csv"

    # 코호트 생성 시도
    cohort = create_cohort(
        admissions_path=ADMISSIONS_FILE_PATH,
        discharge_path=DISCHARGE_FILE_PATH
    )

    print("\n--- Cohort Creation Complete! ---")
    print("Final DataFrame Info:")
    cohort.info()
    print("\nSample Data (first 5 rows):")
    print(cohort.head())

except Exception as e:
    # ❗️ Jupyter 오류 대신 "진짜 오류"를 여기서 출력합니다.
    print("\n" + "="*80)
    print("🚨 [디버깅] '진짜 오류'를 발견했습니다! 🚨")
    print("="*80)
    print(f"오류 유형: {type(e)}")
    print(f"오류 메시지: {e}")
    print("\n--- 전체 Traceback ---")
    traceback.print_exc()
    print("="*80)

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import nltk
nltk.download('punkt_tab')

# NLP 라이브러리
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# 머신러닝 라이브러리
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# NLTK 데이터 다운로드 (최초 1회 실행 필요)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    print("Downloading nltk 'stopwords'...")
    nltk.download('stopwords', quiet=True)

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    print("Downloading nltk 'punkt'...")
    nltk.download('punkt', quiet=True)

print("\n" + "=" * 80)
print("[2단계] 텍스트 전처리 및 피처 엔지니어링")
print("-" * 80)
print(f"시작 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# ============================================================================
# 2.1 텍스트 전처리
# ============================================================================

# NLTK 불용어 세트 로드
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """MIMIC 텍스트 전처리를 위한 함수"""
    if not isinstance(text, str):
        return ""
    
    # 1. 소문자 변환
    text = text.lower()
    
    # 2. 숫자 및 특수문자 제거 (알파벳과 공백만 남김)
    # '[xx...xx]'와 같은 de-identification 태그도 함께 제거
    text = re.sub(r'\[\*\*.*?\*\*\]', ' ', text) # De-id 태그 제거
    text = re.sub(r'[^a-z\s]', ' ', text)       # 알파벳, 공백 외 모두 제거
    
    # 3. 다중 공백을 단일 공백으로 변환
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 4. 토큰화
    tokens = word_tokenize(text)
    
    # 5. 불용어 제거 및 2글자 이하 단어 제거
    processed_tokens = [
        word for word in tokens 
        if word not in stop_words and len(word) > 2
    ]
    
    # 6. 다시 하나의 문자열로 결합
    return ' '.join(processed_tokens)

# 'text' 컬럼에 전처리 함수 적용 (시간이 다소 걸릴 수 있습니다)
print("Preprocessing texts (This may take a few minutes)...")
# .apply() 대신 .map()을 사용하는 것이 Series에서 약간 더 빠릅니다.
cohort['text_clean'] = cohort['text'].map(preprocess_text)

print("  ✓ Text preprocessing complete.")

# 전처리 전후 비교 샘플 출력
print("\n--- Preprocessing Example ---")
print("Original Text (sample):")
print(cohort['text'].iloc[0][:500])
print("\nCleaned Text (sample):")
print(cohort['text_clean'].iloc[0][:500])
print("-----------------------------\n")

# ============================================================================
# 2.2 데이터 분할
# ============================================================================

print("Splitting data into training and test sets (80% / 20%)...")

# 피처(X)와 타겟(y) 정의
X = cohort['text_clean']
y = cohort['hospital_expire_flag']

# 학습용/테스트용 데이터 분할 (8:2)
# stratify=y : 타겟 변수(사망 여부)의 비율을 학습/테스트셋에서 동일하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,    # 20%를 테스트 세트로
    random_state=42,  # 재현 가능성을 위한 시드
    stratify=y        # 타겟 비율 유지
)

print(f"  ✓ Train set: {len(X_train):,} samples")
print(f"    (Mortality rate: {y_train.mean():.2%})")
print(f"  ✓ Test set : {len(X_test):,} samples")
print(f"    (Mortality rate: {y_test.mean():.2%})\n")

# ============================================================================
# 2.3 피처 엔지니어링 (TF-IDF)
# ============================================================================

print("Vectorizing text data using TF-IDF...")

# TF-IDF Vectorizer 초기화
# (5시간 계획에 맞게 속도와 성능을 절충하는 파라미터 사용)
vectorizer = TfidfVectorizer(
    max_features=5000,    # 최대 5,000개의 주요 단어만 피처로 사용
    min_df=5,             # 최소 5개 이상의 문서에 등장한 단어만
    max_df=0.8,           # 80% 이상의 문서에 공통으로 나오는 단어 제외
    ngram_range=(1, 2)    # 단어 1개(unigram) + 단어 2개(bigram) 조합 사용
)

# 1. 학습(Train) 데이터에 대해 fit_transform 수행
# (단어 사전을 학습하고, 텍스트를 벡터로 변환)
X_train_tfidf = vectorizer.fit_transform(X_train)

# 2. 테스트(Test) 데이터에 대해 transform 수행
# (학습된 단어 사전을 기준으로 텍스트를 벡터로 변환)
X_test_tfidf = vectorizer.transform(X_test)

print("  ✓ TF-IDF vectorization complete.")
print(f"  ✓ Vocabulary size: {len(vectorizer.vocabulary_):,} features")
print(f"  ✓ Train data TF-IDF matrix shape: {X_train_tfidf.shape}")
print(f"  ✓ Test data TF-IDF matrix shape : {X_test_tfidf.shape}")

print(f"\n[2단계] 완료. {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

In [ ]:
import pandas as pd
from datetime import datetime

# 머신러닝 모델
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 평가 지표
from sklearn.metrics import (
    roc_auc_score, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)

print("\n" + "=" * 80)
print("[3단계] 모델 학습 및 평가")
print("-" * 80)
print(f"시작 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# ============================================================================
# 3.1 모델 1: 로지스틱 회귀 (Logistic Regression)
# ============================================================================
# - 빠르고, 해석 가능하며, 텍스트 분류에 준수한 성능을 보임
# - class_weight='balanced': 사망(1)/생존(0) 비율이 불균형하므로,
#   적은 쪽(사망)에 더 높은 가중치를 부여하여 학습
# ----------------------------------------------------------------------------

print("[모델 1] 로지스틱 회귀 (Logistic Regression)")
print("Training Logistic Regression model...")

lr_model = LogisticRegression(
    class_weight='balanced', # 불균형 데이터 처리를 위한 핵심 옵션
    random_state=42,
    solver='liblinear',      # 작은 데이터셋 및 이진 분류에 효율적
    max_iter=1000
)

# 모델 학습
lr_model.fit(X_train_tfidf, y_train)

# 예측 수행
# predict: 클래스(0 또는 1) 예측
y_pred_lr = lr_model.predict(X_test_tfidf)
# predict_proba: 클래스별 확률 예측. [:, 1]은 '사망(1)'일 확률
y_proba_lr = lr_model.predict_proba(X_test_tfidf)[:, 1]

print("  ✓ Model training complete.")

# ============================================================================
# 3.2 모델 2: 랜덤 포레스트 (Random Forest)
# ============================================================================
# - 여러 개의 결정 트리를 사용하는 앙상블 모델
# - 비선형 관계도 잘 잡아내며, 일반적으로 강력한 성능을 보임
# ----------------------------------------------------------------------------

print("\n[모델 2] 랜덤 포레스트 (Random Forest)")
print("Training Random Forest model (This may take a minute)...")

rf_model = RandomForestClassifier(
    n_estimators=100,       # 100개의 트리 사용
    class_weight='balanced',# 불균형 데이터 처리
    random_state=42,
    max_depth=15,           # 트리의 최대 깊이 제한 (과적합 방지)
    n_jobs=-1               # 모든 CPU 코어 사용
)

# 모델 학습
rf_model.fit(X_train_tfidf, y_train)

# 예측 수행
y_pred_rf = rf_model.predict(X_test_tfidf)
y_proba_rf = rf_model.predict_proba(X_test_tfidf)[:, 1]

print("  ✓ Model training complete.")

# ============================================================================
# 3.3 모델 평가 및 비교
# ============================================================================

print("\n" + "-" * 80)
print("모델 성능 평가 (Test Set)")
print("-" * 80)

def evaluate_model(y_true, y_pred, y_proba, model_name):
    """모델 성능을 계산하고 딕셔너리로 반환하는 함수"""
    metrics = {
        # AUROC (Area Under ROC): 모델이 0과 1을 얼마나 잘 구별하는지 (핵심 지표)
        'AUROC': roc_auc_score(y_true, y_proba),
        # Accuracy (정확도): 전체 예측 중 맞춘 비율 (불균형 데이터에서는 신뢰도 낮음)
        'Accuracy': accuracy_score(y_true, y_pred),
        # Precision (정밀도): '사망'으로 예측한 것 중 진짜 '사망'인 비율
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        # Recall (재현율): 실제 '사망' 환자 중 모델이 '사망'으로 맞춘 비율
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        # F1-Score: 정밀도와 재현율의 조화 평균
        'F1-Score': f1_score(y_true, y_pred, zero_division=0)
    }
    
    print(f"\n[{model_name}] 평가 지표:")
    for metric, value in metrics.items():
        print(f"  - {metric:<10s}: {value:.4f}")
        
    print(f"\n[{model_name}] Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Survived (0)', 'Expired (1)']))
    
    return metrics

# 각 모델 평가
lr_metrics = evaluate_model(y_test, y_pred_lr, y_proba_lr, "Logistic Regression")
rf_metrics = evaluate_model(y_test, y_pred_rf, y_proba_rf, "Random Forest")

# ============================================================================
# 3.4 결과 요약 (논문용 테이블)
# ============================================================================

print("\n" + "-" * 80)
print("성능 비교 요약 (논문 테이블용)")
print("-" * 80)

# 두 모델의 성능을 하나의 DataFrame으로 만듦
results_df = pd.DataFrame({
    'Logistic Regression': lr_metrics,
    'Random Forest': rf_metrics
}).T  # .T (Transpose)로 행과 열을 바꿈

# AUROC 기준으로 정렬
results_df = results_df.sort_values(by='AUROC', ascending=False)

# 보기 좋게 출력
print(results_df.round(4))

print(f"\n[3단계] 완료. {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

## Exp B — 결과 관련 섹션(Hospital Course, Discharge *, Follow-up) 이후 제거

In [ ]:
import pandas as pd
import sys
import warnings
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

print("\n[1단계 수정] 데이터 로드 및 코호트 생성 (입원 초기 섹션만 사용)")
print("-" * 80)

def load_admissions(filepath):
    """(admissions.csv) 입원 정보 로드"""
    print(f"Loading admissions data from {filepath}...")
    
    usecols = ['subject_id', 'hadm_id', 'hospital_expire_flag', 
               'admittime', 'admission_type', 'admission_location', 'insurance']
    
    dtypes = {
        'subject_id': 'int32',
        'hadm_id': 'int32',
        'hospital_expire_flag': 'int8',
    }
    
    try:
        df = pd.read_csv(filepath, usecols=usecols, dtype=dtypes, parse_dates=['admittime'])
        print(f"  ✓ Loaded {len(df):,} admissions")
        return df
    except FileNotFoundError:
        print(f"\n[오류] {filepath} 파일을 찾을 수 없습니다.")
        sys.exit()

def load_all_notes(filepath):
    """
    (discharge.csv) 임상 노트 로드
    MIMIC-IV-Note v2.2의 discharge.csv 컬럼 구조에 맞게 수정
    """
    print(f"Loading clinical notes from {filepath}...")
    
    # 먼저 파일의 컬럼 확인
    try:
        sample = pd.read_csv(filepath, nrows=0)
        available_columns = sample.columns.tolist()
        print(f"  Available columns: {available_columns}")
    except Exception as e:
        print(f"  Error reading file: {e}")
        sys.exit()
    
    # 필요한 컬럼만 선택
    required_cols = ['subject_id', 'hadm_id', 'text']
    optional_cols = ['charttime', 'storetime']
    
    usecols = required_cols.copy()
    for col in optional_cols:
        if col in available_columns:
            usecols.append(col)
    
    dtypes = {
        'subject_id': 'int32',
        'hadm_id': 'int32',
        'text': 'str'
    }
    
    # 날짜 컬럼 파싱
    parse_dates = [col for col in ['charttime', 'storetime'] if col in usecols]
    
    try:
        if parse_dates:
            df = pd.read_csv(filepath, usecols=usecols, dtype=dtypes, 
                            parse_dates=parse_dates)
        else:
            df = pd.read_csv(filepath, usecols=usecols, dtype=dtypes)
        
        print(f"  ✓ Loaded {len(df):,} clinical notes")
        print(f"  ✓ Columns loaded: {df.columns.tolist()}")
        return df
    except Exception as e:
        print(f"\n[오류] 파일 로드 실패: {e}")
        sys.exit()

def extract_initial_sections(text):
    """
    퇴원 요약본에서 입원 초기 정보만 추출
    (결과가 포함된 섹션 제외)
    """
    if not isinstance(text, str):
        return ""
    
    # 소문자 변환
    text_lower = text.lower()
    
    # 제거할 섹션 (결과 관련)
    sections_to_remove = [
        'hospital course',
        'brief hospital course',
        'discharge disposition',
        'discharge diagnosis',
        'discharge condition',
        'discharge instructions',
        'discharge medications',
        'discharge status',
        'condition on discharge',
        'followup',
        'follow-up',
        'follow up'
    ]
    
    # 제거할 섹션이 시작되는 위치 찾기
    remove_start_idx = len(text)
    
    import re
    for section in sections_to_remove:
        # 섹션 헤더 패턴: "섹션명:" 또는 "섹션명\n" 또는 "___섹션명___"
        patterns = [
            rf'\n\s*{section}\s*[:\n]',
            rf'\n\s*_+\s*{section}\s*_+',
            rf'\n\s*{section.upper()}\s*[:\n]'
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text_lower)
            if match:
                remove_start_idx = min(remove_start_idx, match.start())
    
    # 제거할 섹션 이전까지만 사용
    extracted_text = text[:remove_start_idx]
    
    # 너무 짧으면 원본의 앞부분만이라도 사용 (최대 2000자)
    if len(extracted_text) < 200:
        extracted_text = text[:2000]
    
    return extracted_text

def create_cohort_corrected(admissions_path, notes_path):
    """
    수정된 코호트 생성: 퇴원 요약본에서 입원 초기 섹션만 추출
    """
    
    print("\n[방법] 퇴원 요약본에서 입원 초기 섹션만 추출")
    print("  - Chief Complaint, HPI, PMH, Physical Exam 등만 사용")
    print("  - Hospital Course, Discharge Disposition 등 제외")
    
    # 1. 입원 정보 로드
    df_adm = load_admissions(admissions_path)
    
    # 2. 노트 로드
    df_notes = load_all_notes(notes_path)
    
    # 3. 초기 섹션만 추출
    print("\nExtracting initial sections from discharge summaries...")
    print("  (This may take a few minutes...)")
    
    df_notes['text_initial'] = df_notes['text'].apply(extract_initial_sections)
    
    print("  ✓ Section extraction complete")
    
    # 4. 텍스트 길이 확인
    df_notes['text_length'] = df_notes['text_initial'].str.len()
    print(f"\n  Text length statistics:")
    print(f"    Mean: {df_notes['text_length'].mean():.0f} characters")
    print(f"    Median: {df_notes['text_length'].median():.0f} characters")
    print(f"    Min: {df_notes['text_length'].min():.0f} characters")
    print(f"    Max: {df_notes['text_length'].max():.0f} characters")
    
    # 5. 너무 짧은 텍스트 제거 (초기 섹션이 거의 없는 경우)
    min_length = 200
    before_filter = len(df_notes)
    df_notes = df_notes[df_notes['text_length'] >= min_length]
    after_filter = len(df_notes)
    print(f"\n  Filtered out {before_filter - after_filter:,} notes with length < {min_length}")
    
    # 6. 입원 정보와 병합
    cohort = df_notes[['subject_id', 'hadm_id', 'text_initial']].merge(
        df_adm[['subject_id', 'hadm_id', 'hospital_expire_flag']], 
        on=['subject_id', 'hadm_id'], 
        how='inner'
    )
    
    # 컬럼명 변경
    cohort = cohort.rename(columns={'text_initial': 'text'})
    
    # 7. 중복 및 결측치 제거
    cohort = cohort.drop_duplicates(subset=['hadm_id'], keep='first')
    cohort = cohort.dropna(subset=['text', 'hospital_expire_flag'])
    
    print(f"\n  ✓ Final cohort created: {len(cohort):,} patients")
    print(f"  ✓ Mortality rate: {cohort['hospital_expire_flag'].mean():.2%}")
    
    # 8. 샘플 텍스트 출력 (결과 관련 단어가 있는지 확인)
    print("\n" + "=" * 80)
    print("Sample Text Validation (First 800 characters)")
    print("=" * 80)
    sample_text = cohort['text'].iloc[0][:800]
    print(sample_text)
    print("=" * 80)
    
    # 9. 결과 관련 단어 체크
    print("\n--- Data Leakage Check ---")
    outcome_keywords = ['expired', 'deceased', 'passed away', 'death', 'died', 
                       'comfort care', 'comfort measures', 'palliative', 
                       'withdrawal of care', 'do not resuscitate', 'dnr']
    
    leak_found = False
    for keyword in outcome_keywords:
        count = cohort['text'].str.lower().str.contains(keyword, na=False).sum()
        percentage = (count / len(cohort)) * 100
        if percentage > 5:  # 5% 이상이면 경고
            print(f"  ⚠️  '{keyword}': {count:,} notes ({percentage:.2f}%) - HIGH")
            leak_found = True
        else:
            print(f"  '{keyword}': {count:,} notes ({percentage:.2f}%)")
    
    if leak_found:
        print("\n  ⚠️  WARNING: Significant outcome-related words found!")
        print("     Data leakage may still exist. Consider manual review.")
    else:
        print("\n  ✓ Outcome word frequency is acceptable.")
    
    print("-" * 80)
    
    return cohort

# ============================================================================
# 실행
# ============================================================================
try:
    ADMISSIONS_FILE_PATH = "data/physionet.org/files/mimiciv/3.1/hosp/admissions.csv"
    NOTES_FILE_PATH = "data/physionet.org/files/mimic-iv-note/2.2/note/discharge.csv"

    # 코호트 생성
    cohort = create_cohort_corrected(
        admissions_path=ADMISSIONS_FILE_PATH,
        notes_path=NOTES_FILE_PATH
    )

    print("\n" + "=" * 80)
    print("Cohort Creation Complete!")
    print("=" * 80)
    print("\nDataFrame Info:")
    cohort.info()

    print("\nFirst 3 rows:")
    print(cohort[['subject_id', 'hadm_id', 'hospital_expire_flag']].head(3))
    
    print("\nText length distribution:")
    print(cohort['text'].str.len().describe())

except Exception as e:
    print(f"\n[치명적 오류] 코호트 생성 중 문제가 발생했습니다: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import nltk
nltk.download('punkt_tab')

# NLP 라이브러리
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# 머신러닝 라이브러리
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# NLTK 데이터 다운로드 (최초 1회 실행 필요)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    print("Downloading nltk 'stopwords'...")
    nltk.download('stopwords', quiet=True)

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    print("Downloading nltk 'punkt'...")
    nltk.download('punkt', quiet=True)

print("\n" + "=" * 80)
print("[2단계] 텍스트 전처리 및 피처 엔지니어링")
print("-" * 80)
print(f"시작 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# ============================================================================
# 2.1 텍스트 전처리
# ============================================================================

# NLTK 불용어 세트 로드
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """MIMIC 텍스트 전처리를 위한 함수"""
    if not isinstance(text, str):
        return ""
    
    # 1. 소문자 변환
    text = text.lower()
    
    # 2. 숫자 및 특수문자 제거 (알파벳과 공백만 남김)
    # '[xx...xx]'와 같은 de-identification 태그도 함께 제거
    text = re.sub(r'\[\*\*.*?\*\*\]', ' ', text) # De-id 태그 제거
    text = re.sub(r'[^a-z\s]', ' ', text)       # 알파벳, 공백 외 모두 제거
    
    # 3. 다중 공백을 단일 공백으로 변환
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 4. 토큰화
    tokens = word_tokenize(text)
    
    # 5. 불용어 제거 및 2글자 이하 단어 제거
    processed_tokens = [
        word for word in tokens 
        if word not in stop_words and len(word) > 2
    ]
    
    # 6. 다시 하나의 문자열로 결합
    return ' '.join(processed_tokens)

# 'text' 컬럼에 전처리 함수 적용 (시간이 다소 걸릴 수 있습니다)
print("Preprocessing texts (This may take a few minutes)...")
# .apply() 대신 .map()을 사용하는 것이 Series에서 약간 더 빠릅니다.
cohort['text_clean'] = cohort['text'].map(preprocess_text)

print("  ✓ Text preprocessing complete.")

# 전처리 전후 비교 샘플 출력
print("\n--- Preprocessing Example ---")
print("Original Text (sample):")
print(cohort['text'].iloc[0][:500])
print("\nCleaned Text (sample):")
print(cohort['text_clean'].iloc[0][:500])
print("-----------------------------\n")

# ============================================================================
# 2.2 데이터 분할
# ============================================================================

print("Splitting data into training and test sets (80% / 20%)...")

# 피처(X)와 타겟(y) 정의
X = cohort['text_clean']
y = cohort['hospital_expire_flag']

# 학습용/테스트용 데이터 분할 (8:2)
# stratify=y : 타겟 변수(사망 여부)의 비율을 학습/테스트셋에서 동일하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,    # 20%를 테스트 세트로
    random_state=42,  # 재현 가능성을 위한 시드
    stratify=y        # 타겟 비율 유지
)

print(f"  ✓ Train set: {len(X_train):,} samples")
print(f"    (Mortality rate: {y_train.mean():.2%})")
print(f"  ✓ Test set : {len(X_test):,} samples")
print(f"    (Mortality rate: {y_test.mean():.2%})\n")

# ============================================================================
# 2.3 피처 엔지니어링 (TF-IDF)
# ============================================================================

print("Vectorizing text data using TF-IDF...")

# TF-IDF Vectorizer 초기화
# (5시간 계획에 맞게 속도와 성능을 절충하는 파라미터 사용)
vectorizer = TfidfVectorizer(
    max_features=5000,    # 최대 5,000개의 주요 단어만 피처로 사용
    min_df=5,             # 최소 5개 이상의 문서에 등장한 단어만
    max_df=0.8,           # 80% 이상의 문서에 공통으로 나오는 단어 제외
    ngram_range=(1, 2)    # 단어 1개(unigram) + 단어 2개(bigram) 조합 사용
)

# 1. 학습(Train) 데이터에 대해 fit_transform 수행
# (단어 사전을 학습하고, 텍스트를 벡터로 변환)
X_train_tfidf = vectorizer.fit_transform(X_train)

# 2. 테스트(Test) 데이터에 대해 transform 수행
# (학습된 단어 사전을 기준으로 텍스트를 벡터로 변환)
X_test_tfidf = vectorizer.transform(X_test)

print("  ✓ TF-IDF vectorization complete.")
print(f"  ✓ Vocabulary size: {len(vectorizer.vocabulary_):,} features")
print(f"  ✓ Train data TF-IDF matrix shape: {X_train_tfidf.shape}")
print(f"  ✓ Test data TF-IDF matrix shape : {X_test_tfidf.shape}")

print(f"\n[2단계] 완료. {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

In [ ]:
import pandas as pd
from datetime import datetime

# 머신러닝 모델
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 평가 지표
from sklearn.metrics import (
    roc_auc_score, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)

print("\n" + "=" * 80)
print("[3단계] 모델 학습 및 평가")
print("-" * 80)
print(f"시작 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# ============================================================================
# 3.1 모델 1: 로지스틱 회귀 (Logistic Regression)
# ============================================================================
# - 빠르고, 해석 가능하며, 텍스트 분류에 준수한 성능을 보임
# - class_weight='balanced': 사망(1)/생존(0) 비율이 불균형하므로,
#   적은 쪽(사망)에 더 높은 가중치를 부여하여 학습
# ----------------------------------------------------------------------------

print("[모델 1] 로지스틱 회귀 (Logistic Regression)")
print("Training Logistic Regression model...")

lr_model = LogisticRegression(
    class_weight='balanced', # 불균형 데이터 처리를 위한 핵심 옵션
    random_state=42,
    solver='liblinear',      # 작은 데이터셋 및 이진 분류에 효율적
    max_iter=1000
)

# 모델 학습
lr_model.fit(X_train_tfidf, y_train)

# 예측 수행
# predict: 클래스(0 또는 1) 예측
y_pred_lr = lr_model.predict(X_test_tfidf)
# predict_proba: 클래스별 확률 예측. [:, 1]은 '사망(1)'일 확률
y_proba_lr = lr_model.predict_proba(X_test_tfidf)[:, 1]

print("  ✓ Model training complete.")

# ============================================================================
# 3.2 모델 2: 랜덤 포레스트 (Random Forest)
# ============================================================================
# - 여러 개의 결정 트리를 사용하는 앙상블 모델
# - 비선형 관계도 잘 잡아내며, 일반적으로 강력한 성능을 보임
# ----------------------------------------------------------------------------

print("\n[모델 2] 랜덤 포레스트 (Random Forest)")
print("Training Random Forest model (This may take a minute)...")

rf_model = RandomForestClassifier(
    n_estimators=100,       # 100개의 트리 사용
    class_weight='balanced',# 불균형 데이터 처리
    random_state=42,
    max_depth=15,           # 트리의 최대 깊이 제한 (과적합 방지)
    n_jobs=-1               # 모든 CPU 코어 사용
)

# 모델 학습
rf_model.fit(X_train_tfidf, y_train)

# 예측 수행
y_pred_rf = rf_model.predict(X_test_tfidf)
y_proba_rf = rf_model.predict_proba(X_test_tfidf)[:, 1]

print("  ✓ Model training complete.")

# ============================================================================
# 3.3 모델 평가 및 비교
# ============================================================================

print("\n" + "-" * 80)
print("모델 성능 평가 (Test Set)")
print("-" * 80)

def evaluate_model(y_true, y_pred, y_proba, model_name):
    """모델 성능을 계산하고 딕셔너리로 반환하는 함수"""
    metrics = {
        # AUROC (Area Under ROC): 모델이 0과 1을 얼마나 잘 구별하는지 (핵심 지표)
        'AUROC': roc_auc_score(y_true, y_proba),
        # Accuracy (정확도): 전체 예측 중 맞춘 비율 (불균형 데이터에서는 신뢰도 낮음)
        'Accuracy': accuracy_score(y_true, y_pred),
        # Precision (정밀도): '사망'으로 예측한 것 중 진짜 '사망'인 비율
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        # Recall (재현율): 실제 '사망' 환자 중 모델이 '사망'으로 맞춘 비율
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        # F1-Score: 정밀도와 재현율의 조화 평균
        'F1-Score': f1_score(y_true, y_pred, zero_division=0)
    }
    
    print(f"\n[{model_name}] 평가 지표:")
    for metric, value in metrics.items():
        print(f"  - {metric:<10s}: {value:.4f}")
        
    print(f"\n[{model_name}] Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Survived (0)', 'Expired (1)']))
    
    return metrics

# 각 모델 평가
lr_metrics = evaluate_model(y_test, y_pred_lr, y_proba_lr, "Logistic Regression")
rf_metrics = evaluate_model(y_test, y_pred_rf, y_proba_rf, "Random Forest")

# ============================================================================
# 3.4 결과 요약 (논문용 테이블)
# ============================================================================

print("\n" + "-" * 80)
print("성능 비교 요약 (논문 테이블용)")
print("-" * 80)

# 두 모델의 성능을 하나의 DataFrame으로 만듦
results_df = pd.DataFrame({
    'Logistic Regression': lr_metrics,
    'Random Forest': rf_metrics
}).T  # .T (Transpose)로 행과 열을 바꿈

# AUROC 기준으로 정렬
results_df = results_df.sort_values(by='AUROC', ascending=False)

# 보기 좋게 출력
print(results_df.round(4))

print(f"\n[3단계] 완료. {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

## 결과 시각화 (ROC · PR · Confusion Matrix · Top 피처)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from datetime import datetime

# 평가 지표 (시각화를 위해)
from sklearn.metrics import (
    roc_curve, 
    precision_recall_curve, 
    auc,
    confusion_matrix
)

# 시각화 한글/마이너스 폰트 설정 (필요한 경우)
# plt.rcParams['font.family'] = 'AppleGothic' # Mac
# plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

# 시각화 스타일 설정
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n" + "=" * 80)
print("[4단계] 결과 분석 및 시각화")
print("-" * 80)
print(f"시작 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# ============================================================================
# 4.1 시각화 (2x2 플롯 생성)
# ============================================================================
# 논문에 포함할 핵심 그림 4개를 하나의 Figure로 만듭니다.

print("Generating plots (ROC, PR, Confusion Matrix, Feature Importance)...")

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('MIMIC-IV 사망률 예측 모델 결과 분석', fontsize=20, fontweight='bold')

# --- 1. ROC Curve (좌상단) ---
ax1 = axes[0, 0]
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
auroc_lr = lr_metrics['AUROC']
auroc_rf = rf_metrics['AUROC']

ax1.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auroc_lr:.4f})', lw=2)
ax1.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auroc_rf:.4f})', lw=2)
ax1.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.5)')
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('ROC Curve', fontsize=15, fontweight='bold')
ax1.legend()

# --- 2. Precision-Recall Curve (우상단) ---
ax2 = axes[0, 1]
prec_lr, rec_lr, _ = precision_recall_curve(y_test, y_proba_lr)
prec_rf, rec_rf, _ = precision_recall_curve(y_test, y_proba_rf)
pr_auc_lr = auc(rec_lr, prec_lr)
pr_auc_rf = auc(rec_rf, prec_rf)

ax2.plot(rec_lr, prec_lr, label=f'Logistic Regression (PR-AUC = {pr_auc_lr:.4f})', lw=2)
ax2.plot(rec_rf, prec_rf, label=f'Random Forest (PR-AUC = {pr_auc_rf:.4f})', lw=2)
ax2.set_xlabel('Recall (Sensitivity)')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve', fontsize=15, fontweight='bold')
ax2.legend()

# --- 3. Confusion Matrix - Logistic Regression (좌하단) ---
# (해석이 더 용이한 로지스틱 회귀 모델을 기준으로 그림)
ax3 = axes[1, 0]
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=ax3, cbar=False,
            annot_kws={"size": 14})
ax3.set_title('Confusion Matrix (Logistic Regression)', fontsize=15, fontweight='bold')
ax3.set_xlabel('Predicted Label')
ax3.set_ylabel('True Label')
ax3.set_xticklabels(['Survived (0)', 'Expired (1)'])
ax3.set_yticklabels(['Survived (0)', 'Expired (1)'])

# --- 4. Feature Importance - Logistic Regression (우하단) ---
# (로지스틱 회귀의 계수(coefficient)를 통해 어떤 단어가 예측에 영향을 미쳤는지 확인)
ax4 = axes[1, 1]
try:
    feature_names = vectorizer.get_feature_names_out()
    coefficients = lr_model.coef_[0]
    
    # 계수를 데이터프레임으로 만듦
    coef_df = pd.DataFrame({'feature': feature_names, 'coef': coefficients})
    
    # 상위 15개 (사망 관련) + 하위 15개 (생존 관련)
    top_n = 15
    plot_df = pd.concat([
        coef_df.nlargest(top_n, 'coef'),
        coef_df.nsmallest(top_n, 'coef')
    ]).sort_values('coef', ascending=False)

    # 색상 지정
    colors = ['#d62728' if c > 0 else '#2ca02c' for c in plot_df['coef']]
    
    sns.barplot(data=plot_df, x='coef', y='feature', palette=colors, ax=ax4)
    ax4.set_title('Top Feature Importance (Logistic Regression)', fontsize=15, fontweight='bold')
    ax4.set_xlabel('Coefficient (Weight)')
    ax4.set_ylabel('Feature (Word)')
    ax4.axvline(0, color='k', linestyle='--', lw=1)
    
except Exception as e:
    ax4.text(0.5, 0.5, f"""Feature Importance  
Error: {e}""", # <--- f"""..."""로 수정
             ha='center', va='center', color='red')


# 전체 그림 저장
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Main Title과 겹치지 않게
plt.savefig('mortality_prediction_results.png', dpi=300)

print("  ✓ Plots generated and saved to 'mortality_prediction_results.png'")
# plt.show() # Jupyter Notebook 환경에서는 이 줄을 실행하면 그림이 보임

# ============================================================================
# 4.2 논문 작성용 요약 통계
# ============================================================================

print("\n" + "=" * 80)
print("KSC 2025 논문 작성용 요약")
print("=" * 80)

# 1단계의 cohort 변수에서 통계 추출 (만약 cohort가 없다면 1단계 재실행 필요)
try:
    total_patients = len(cohort)
    expired_patients = cohort['hospital_expire_flag'].sum()
    mortality_rate = cohort['hospital_expire_flag'].mean()
except NameError:
    print("[Warning] 'cohort' 변수를 찾을 수 없습니다. 1단계를 재실행하세요.")
    total_patients, expired_patients, mortality_rate = (np.nan, np.nan, np.nan)

# 3단계의 results_df 변수에서 성능 추출
try:
    best_model_name = results_df.index[0]
    best_auroc = results_df.iloc[0]['AUROC']
except NameError:
    print("[Warning] 'results_df' 변수를 찾을 수 없습니다. 3단계를 재실행하세요.")
    best_model_name, best_auroc = ("N/A", np.nan)

# 4단계의 특성 중요도에서 단어 추출
try:
    mortality_words = plot_df[plot_df['coef'] > 0]['feature'].tolist()
    survival_words = plot_df[plot_df['coef'] < 0]['feature'].tolist()
except Exception:
    mortality_words, survival_words = ([], [])

# --- 논문 초록/결과 섹션에 사용할 요약본 ---
summary_text = f"""
## KSC 2025 논문 핵심 결과 요약

### [표 1] 코호트 기본 통계
* **데이터:** MIMIC-IV-Note (v2.2)
* **대상:** 퇴원 요약본(Discharge Summary)이 있는 입원 건
* **총 환자 수:** {total_patients:,.0f} 명
* **병원 내 사망:** {expired_patients:,.0f} 명 (사망률: {mortality_rate:.2%})

### [표 2] 모델 성능 비교 (Test Set)
{results_df.round(4).to_string()}

### [그림 1] 주요 결과 시각화
* (mortality_prediction_results.png 파일 참조)

### 핵심 결과 (Results)
본 연구에서는 MIMIC-IV 퇴원 요약본 텍스트를 TF-IDF (n-gram: 1,2, max_features: 5000)로 벡터화하고, 두 머신러닝 모델을 사용하여 병원 내 사망률을 예측하였다.

1.  **모델 성능:** {best_model_name} 모델이 **AUROC {best_auroc:.4f}**로 가장 우수한 성능을 보였다. (Logistic Regression AUROC: {lr_metrics['AUROC']:.4f}, Random Forest AUROC: {rf_metrics['AUROC']:.4f})
2.  **주요 예측 인자 (사망 관련):** 로지스틱 회귀 분석 결과, '{', '.join(mortality_words[:3])}' 등과 같은 단어가 사망 예측과 높은 양의 상관관계를 보였다.
3.  **주요 예측 인자 (생존 관련):** 반면, '{', '.join(survival_words[-3:])}' 등과 같은 단어는 생존 예측과 높은 음의 상관관계를 보였다.

### 고찰 및 한계점 (Discussion)
* 간단한 TF-IDF와 고전적 머신러닝 모델만으로도 임상 텍스트에서 유의미한 예측 성능(AUROC ~{best_auroc:.2f})을 확보할 수 있음을 확인했다.
* 이는 임상 노트의 비정형 텍스트가 환자의 예후를 반영하는 중요한 정보를 포함하고 있음을 시사한다.
* **한계점:**
    1.  TF-IDF는 단어의 문맥적 의미를 파악하지 못함.
    2.  최신 딥러닝 기반 언어 모델(예: ClinicalBERT)을 사용하지 않음.
    3.  퇴원 요약본 외 다른 노트(간호 기록 등)를 활용하지 않음.

### 결론 (Conclusion)
본 예비 분석(preliminary analysis)을 바탕으로, 향후 연구에서는 ClinicalBERT와 같은 사전 학습 모델을 적용하고 다양한 임상 노트를 통합하여 예측 성능을 고도화할 필요가 있다.
"""

print(summary_text)

# 요약본 파일로 저장
with open('KSC_paper_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary_text)

print(f"\n  ✓ 논문 요약 저장 완료: 'KSC_paper_summary.txt'")
print(f"\n[4단계] 완료. 5시간 논문 프로젝트의 모든 코드가 실행되었습니다.")
print(f"완료 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)